# 04 — Parse the Datasheet: regex vs. the Document Parser API

Companion to [Chapter 10](../10-datasheet-parsing.md). Runs **both** techniques
against the same file so you can compare their output directly. Technique 2's cell
order: auth -> submit `start` -> poll `byids` with a time budget -> inspect the rich
job detail -> `write` -> verify the view node.

Before running Technique 2, make sure you've completed the description-engineering
step (Chapter 10, section 10.3) and redeployed your `data_modeling` module.

In [ ]:
YOURNAME = "YOURNAME"  # [CHANGE]

import io
import re
import time
from datetime import datetime, timezone

from cognite.client import CogniteClient
from cognite.client.data_classes.data_modeling import (
    DirectRelationReference, NodeApply, NodeId, NodeOrEdgeData, ViewId,
)
from pypdf import PdfReader

client = CogniteClient()
space = f"isp_{YOURNAME}_TRN"
schema_edm = f"ssp_{YOURNAME}_TrainingCore_edm"
schema_sdm = f"ssp_{YOURNAME}_MaintenanceInsight_sdm"
model_version = "v1.0.0"
file_xid = f"file_{YOURNAME}_TRN_DS_21_PA_2001A"

## Technique 1 — deterministic regex

Extract text with `pypdf`, match a fixed `PATTERNS` dict. This is your baseline --
look at exactly what it finds and what it lists as `missing`.

In [ ]:
PATTERNS = {
    "ratedFlowM3h": r"Rated Flow:\s*([\d.]+)\s*m3/h",
    "ratedHeadM": r"Rated Head:\s*([\d.]+)\s*m",
    "ratedPowerKw": r"Rated Power:\s*([\d.]+)\s*kW",
    "designPressureBarg": r"Design Pressure:\s*([\d.]+)\s*barg",
    "designTemperatureC": r"Design Temperature:\s*([\d.]+)\s*degC",
    "dryWeightKg": r"Dry Weight:\s*([\d.]+)\s*kg",
    "casingMaterial": r"Casing Material:\s*(.+)",
    "sealType": r"Seal Type:\s*(.+)",
}
NUMERIC_KEYS = {"ratedFlowM3h", "ratedHeadM", "ratedPowerKw", "designPressureBarg", "designTemperatureC", "dryWeightKg"}

content = client.files.download_bytes(instance_id=NodeId(space, file_xid))
reader = PdfReader(io.BytesIO(content))
text = "\n".join(page.extract_text() or "" for page in reader.pages)

regex_parsed, regex_missing = {}, []
for key, pattern in PATTERNS.items():
    m = re.search(pattern, text)
    if not m:
        regex_missing.append(key)
        continue
    raw = m.group(1).strip()
    regex_parsed[key] = float(raw) if key in NUMERIC_KEYS else raw

print("regex parsed:", regex_parsed)
print("regex missing:", regex_missing)

## Technique 2 — Document Parser API: `start`

This endpoint has no typed SDK method -- raw `client.post`. `viewConfig` points at
your EHP view; the view's property names + descriptions ARE the extraction schema.

In [ ]:
DOCPARSER = f"/api/v1/projects/{client.config.project}/context/documentparser"

# /jobs/start is single-job (flat body). Batch uses POST /jobs with items[].
start_body = {
    "viewConfig": {"space": schema_sdm, "externalId": "viw_EquipmentHealthProfile_sdm", "version": model_version},
    "files": [{"fileInstanceId": {"space": space, "externalId": file_xid}}],
    "node": {"space": space, "externalId": "ehp_21-PA-2001A"},
    "useVision": True,
    "userPrompt": (
        "Extract pump datasheet specifications per the target view's property "
        "descriptions. If a value is not explicitly present, leave it empty -- do not guess."
    ),
}
start_resp = client.post(f"{DOCPARSER}/jobs/start", json=start_body).json()
job_id = start_resp["jobId"]
print("jobId:", job_id, "initial status:", start_resp.get("status"))


## Poll via `byids` -- NEVER the single `GET /{jobId}`

`[COMMON MISTAKE]` the single-job GET is unreliable / 404s in practice on this API.
Always poll through `POST /jobs/byids`, and always with a time budget -- this is an
async job; never block indefinitely on a result.

In [ ]:
deadline = time.time() + 8 * 60
status = "Queued"
detail = None
while status in ("Queued", "Running") and time.time() < deadline:
    time.sleep(15)
    byids_resp = client.post(f"{DOCPARSER}/jobs/byids", json={"items": [{"jobId": job_id}]}).json()
    detail = byids_resp["items"][0]
    status = detail["status"]["job"] if isinstance(detail.get("status"), dict) else detail.get("status")
    print("status:", status)

print("final status:", status)

## Inspect the rich job detail before writing

This is the notebook's value-add over calling the Function blind: per-field
confidence scores and spatial bounding boxes, plus view/validation status.

In [ ]:
result = detail.get("result") or {}
print("view status:", detail["status"].get("view"))
print("validation:", detail["status"].get("validation"))
print("scores:", result.get("scores"))
for prop, answer in (result.get("rawResponses") or {}).items():
    print(f"  {prop}: value={answer.get('value')!r} page={answer.get('pageNum')} spatialData={answer.get('spatialData')}")


## `write` -- commit the parsed answers into the view node

In [ ]:
assert status == "Completed", f"job is {status}, not Completed -- do not call write yet"

try:
    client.post(f"{DOCPARSER}/jobs/write", json={"items": [{"jobId": job_id}]})
    print("jobs/write ok")
except Exception as exc:
    print(f"jobs/write failed ({exc}) -- writing extracted fields via instances.apply")

FLOAT_FIELDS = ["ratedFlowM3h", "ratedHeadM", "ratedPowerKw",
                "designPressureBarg", "designTemperatureC", "dryWeightKg"]
TEXT_FIELDS = ["casingMaterial", "sealType"]
raw = result.get("rawResponses") or {}
props = {}
for f in FLOAT_FIELDS:
    v = (raw.get(f) or {}).get("value")
    if v not in (None, ""):
        props[f] = float(str(v).split()[0])   # tolerate "320 m3/h"
for f in TEXT_FIELDS:
    v = (raw.get(f) or {}).get("value")
    if v not in (None, ""):
        props[f] = str(v)

v_ehp = ViewId(schema_sdm, "viw_EquipmentHealthProfile_sdm", "v1.0.0")
client.data_modeling.instances.apply(nodes=[NodeApply(
    space=space, external_id="ehp_21-PA-2001A",
    sources=[NodeOrEdgeData(source=v_ehp, properties=props)],
)])
print("wrote:", sorted(props))


## Verify, and compare against Technique 1

In [ ]:
v_ehp = ViewId(schema_sdm, "viw_EquipmentHealthProfile_sdm", model_version)
node = client.data_modeling.instances.retrieve_nodes(nodes=[(space, "ehp_21-PA-2001A")], sources=[v_ehp])[0]
api_parsed = node.properties.get(v_ehp, {})

print("Document Parser API result:", api_parsed)
print("\nSide by side on shared fields:")
for key in NUMERIC_KEYS | {"casingMaterial", "sealType"}:
    print(f"  {key}: regex={regex_parsed.get(key)!r}  api={api_parsed.get(key)!r}")

## Cleanup note

Unlike entity-matching models (notebook 01), Document Parser jobs are **not** global
schema objects that pollute a shared namespace -- there is no mandatory delete step
here. `DELETE /context/documentparser/{jobId}` exists for teardown/hygiene but is
optional for this lab.

## Bridge to the Function

Package Technique 2 into `ParseDatasheet`: env vars instead of notebook variables,
the same bounded-poll discipline, and an added step the PDF text alone can never
give you -- computing `openWorkOrderCount` from your work-order nodes and writing the
`asset`/`equipment`/`datasheetFile` relations yourself. See
[Chapter 10, section 10.5](../10-datasheet-parsing.md#105-write-the-function-parsedatasheet-technique-2-deployed).